<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Speech Commands Keyword Spotting (KWS) — Akida 2 Benchmark

Run Time: ~10 minutes

This notebook walks through evaluating and benchmarking a model on the **Speech Commands** dataset, targeting Akida 2 hardware.

For details on the dataset and preparation of the model, see the neighbouring README.md and speech_commands_notebook_training.ipynb.

> **Note:** the hardware benchmark section needs a physical Akida 2 FPGA device. Everything is guarded, so the notebook runs end-to-end without a board — the hardware cells simply skip. It will not produce hardware latency numbers in Colab.

## Model

A pretrained Akida model is included in this repo, at `pretrained_models/ds_cnn_speech_commands_i8_w4_a4_qat.fbz`. As with all model files in the repo, that is handled via `git-lfs` (for efficient large file storage). If you haven't set that up yet, see the [Trained models](../../../README.md#trained-models) section of the top-level README.

If you have run through the training scripts or notebook, and prefer to benchmark the Akida model generated through that, simply modify the model directory and filename in the following cell (point `MODELS_DIR` at `./models/`).

In [ ]:
import os
import numpy as np
import akida

from brainchip_utils.hardware_utils import AKIDA_CLOCKS_HZ

DATA_PATH = './data/speech_commands'
MODELS_DIR = './pretrained_models/'
INPUT_SHAPE = (49, 10, 1)

# Which converted model to benchmark. The training pipeline produces two variants:
#   ds_cnn_speech_commands_i8_w8_a8.fbz     (8-bit)
#   ds_cnn_speech_commands_i8_w4_a4_qat.fbz (4-bit QAT)
MODEL_FILENAME = 'ds_cnn_speech_commands_i8_w4_a4_qat.fbz'
MODEL_FBZ = os.path.join(MODELS_DIR, MODEL_FILENAME)

# Clocks (see speech_commands_benchmark.py and brainchip_utils.hardware_utils.AKIDA_CLOCKS_HZ).
MEASURED_CLOCK = AKIDA_CLOCKS_HZ['AKIDA2_FPGA']  # 25 MHz FPGA
PROJECTED_CLOCK = AKIDA_CLOCKS_HZ['AKD2500']     # 1 GHz target (pre-production)

To load the model, we simply pass the path to the `.fbz` file to the `akida.Model()` method:

In [ ]:
ak_model = akida.Model(MODEL_FBZ)
ak_model.summary()

## Device

`get_akida_device` returns `None` when no compatible hardware is present, in which case the benchmark below is skipped.

In [ ]:
from brainchip_utils.hardware_utils import get_akida_device

device = get_akida_device(target_version=ak_model.ip_version)
if device is None:
    print('No compatible Akida hardware device found — benchmark will be skipped.')
else:
    print('Akida device found:', device)

## Samples

Akida latency is activity-dependent (it exploits sparsity), so we benchmark on real inputs rather than random data.

In [ ]:
from speech_commands_data import get_samples, compute_mfcc_range

NUM_SAMPLES = 100
# MFCC uint8 scaling bounds from the train split (same as training/eval).
data_transform = compute_mfcc_range(data_dir=DATA_PATH)
samples = get_samples(DATA_PATH, data_transform=data_transform, num_samples=NUM_SAMPLES)

## Latency benchmark

Full-model benchmark in each mapping mode, with measured (25 MHz) and projected latency. Cycle count is fixed for a given model + mapping, so `projected_ms = mean_inf_clk / PROJECTED_CLOCK * 1000`.

In [ ]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats

if device is not None:
    for mm in ['Minimal', 'AllNps']:
        map_mode = getattr(akida.MapMode, mm)
        res = full_model_benchmark(ak_model, device, samples,
                                   map_mode=map_mode, clock_freq=MEASURED_CLOCK)
        projected_ms = res['mean_inf_clk'] / PROJECTED_CLOCK * 1000
        ak_model.map(device, mode=map_mode)
        num_nps, num_passes, num_sequences = get_mapping_stats(ak_model)
        print(f'[{mm}] NPs={num_nps} passes={num_passes} '
              f'latency@25MHz={res["mean_clk_ms"]:.3f} ms '
              f'projected@1000MHz={projected_ms:.3f} ms')
else:
    print('Hardware not available — skipping latency benchmark.')

## Per-layer benchmark & sparsity

Per-layer latency (Minimal mapping) plus activation sparsity, which drives Akida efficiency.

In [ ]:
from brainchip_utils.hardware_utils import per_layer_benchmark
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity

if device is not None:
    ak_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    per_layer_results = per_layer_benchmark(ak_model, device, samples,
                                            repeats=NUM_SAMPLES, clock_freq=MEASURED_CLOCK)
    print('Per-layer benchmark complete.')
else:
    # Sparsity can still be computed on the software backend without hardware.
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    print('Hardware not available — skipped latency; sparsity computed on software backend.')